In [0]:
%run ./04_Function_Book

In [0]:
# Step 1: Define the configuration object specifying all data quality rules for customers.
# Includes null checks, regex/format, range limits, allowable values (enum), and PK uniqueness.
dq_config = {
    "table_name": "customers",

    # NULL CHECKS
    "null_checks": {
        "customer_id": 0,
        "customer_unique_id": 0,
        "customer_zip_code_prefix": 0,
        "customer_city": 0,
        "customer_state": 0
    },

    # FORMAT CHECKS
    "format_checks": {

        # REGEX CHECK
        "regex": {
            "customer_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            },
            "customer_unique_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            },
            "customer_city": {
                "pattern": r"^[A-Za-z0-9\s\-\']+$",
                "threshold": 0
            },
            "customer_state": {
                "pattern": r"^[A-Za-z]{2}$",
                "threshold": 0
            }
        },
    },

    # RANGE CHECKS
    "range_checks": {
        "customer_zip_code_prefix": {
            "min": 1003,
            "max": 99990,
            "threshold": 0
        }
    },

    # ENUM CHECKS
    "enum_checks": {
        "customer_state": {
            "allowed_values": [
                'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT',
                'MS','MG','PA','PB','PR','PE','PI','RJ','RN','RS',
                'RO','RR','SC','SP','SE','TO'
            ],
            "threshold": 0
        }
    },

    # PRIMARY KEY
    "primary_key": {
        "column": "customer_id", 
        "threshold": 0
    }
}

In [0]:
# Define a function to generate metrics DataFrame from the Spark DataFrame and config
# Applies null/format/range/enum/pk checks and aggregates results with a timestamp
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

def generate_metrics_df(df, config):
    all_results = []
    all_results += run_null_checks(df, config)
    all_results += run_format_checks(df, config)
    all_results += run_range_checks(df, config)
    all_results += run_enum_checks(df, config)
    all_results += run_pk_check(df, config)
    for result in all_results:
        result["table_name"] = config["table_name"]
    metrics_df = spark.createDataFrame(Row(**r) for r in all_results) \
        .withColumn("run_time", current_timestamp())
    return metrics_df

In [0]:
# Load the customers silver table and apply the metrics generator
# This runs all validation checks on actual customer data
df_customers = spark.table("retail_catalog.silver.customers")
metrics_df = generate_metrics_df(df_customers, dq_config)

In [0]:
# Return calculated metrics DataFrame for collection in the orchestrating/master notebook
metrics_df

DataFrame[column_name: string, check_name: string, metric_type: string, metric_value: bigint, threshold: bigint, status: string, table_name: string, run_time: timestamp]